# Torax Simulations

Thanks to the hard work of the Torax team in making a well-documented and easy to use library, we can run Torax quite easily from a Jupyter Notebook. Here is an example pulling TRANSP profile and CFSPOPCON data for the SPARC PRD to build a configuration dictionary for Torax.

Torax simulations are configured in a dictionary form. We recommend referring to the [Torax documentation](https://torax.readthedocs.io/en/latest/) to determine how to configure your simulations.

In [ ]:
%load_ext autoreload
%autoreload 2
from popsim.interfaces.cfspopcon_scenario import load_cfspopcon_scenario_for_comet_mirror
from popsim.interfaces.sparc_public import load_prd_transp_profiles
from popsim.interfaces.torax import run_torax

transp_data = load_prd_transp_profiles()
input_parameters, species_container, species_concentrations = load_cfspopcon_scenario_for_comet_mirror("SPARC_PRD")

ped_top = 0.95  # Location of the pedestal top in normalized radius.

# Configuration adapted from the Torax iterhybrid_predictor_corrector example.
CONFIG = {
    "runtime_params": {
        "plasma_composition": {
            # physical inputs
            "Ai": 2.5,  # amu of main ion (if multiple isotope, make average)
            "Zeff": 1.5,  # needed for qlknn and fusion power
            # effective impurity charge state.
            "Zimp": 10,
        },
        "profile_conditions": {
            "Ip_tot": 8.7,  # total plasma current in MA
            # boundary + initial conditions for T and n
            # initial condition ion temperature for r=0 and r=Rmin
            "Ti": {0.0: {0.0: 15.0, 1.0: 0.2}},
            "Ti_bound_right": float(transp_data["Ti_keV"].sel(rho=1.0).values),  # boundary condition ion temperature for r=Rmin
            # initial condition electron temperature for r=0 and r=Rmin
            "Te": {0.0: {0.0: 15.0, 1.0: 0.2}},
            "Te_bound_right": float(transp_data["Te_keV"].sel(rho=1.0).values),  # boundary condition electron temp for r=Rmin
            "ne_bound_right": float(transp_data["ne20"].sel(rho=1.0).values),  # boundary condition density for r=Rmin
            # set initial condition density according to Greenwald fraction.
            "ne_is_fGW": True,
            "nbar": 0.37,
            "ne": {0: {0.0: 1.5, 1.0: 1.0}},  # Initial electron density profile
            # internal boundary condition (pedestal)
            # do not set internal boundary condition if this is False
            "set_pedestal": True,
            "Tiped": float(transp_data["Ti_keV"].sel(rho=ped_top).values),  # ion pedestal top temperature in keV for Ti and Te
            "Teped": float(transp_data["Te_keV"].sel(rho=ped_top).values),  # electron pedestal top temperature in keV for Ti and Te
            "neped": float(transp_data["ne20"].sel(rho=ped_top).values),  # pedestal top electron density in units of nref
            "Ped_top": ped_top,  # set ped top location in normalized radius
        },
        "numerics": {
            # simulation control
            "t_final": 5,  # length of simulation time in seconds
            # 1/multiplication factor for sigma (conductivity) to reduce current
            # diffusion timescale to be closer to heat diffusion timescale.
            "resistivity_mult": 200,
            "ion_heat_eq": True,
            "el_heat_eq": True,
            "current_eq": True,
            "dens_eq": True,
            "maxdt": 0.1,
            # multiplier in front of the base timestep dt=dx^2/(2*chi). Can
            # likely be increased further beyond this default.
            "dtmult": 50,
            "dt_reduction_factor": 3,
        },
    },
    "geometry": {
        "geometry_type": "circular",
        "elongation_LCFS": 1.97,
        "Rmaj": input_parameters["major_radius"],  # major radius (R) in meters
        "Rmin": input_parameters["inverse_aspect_ratio"] * input_parameters["major_radius"],  # minor radius (a) in meters
        "B0": input_parameters["magnetic_field_on_axis"],  # Toroidal magnetic field on axis [T]
    },
    "sources": {
        # Current sources (for psi equation)
        "j_bootstrap": {
            # Multiplication factor for bootstrap current.
            "bootstrap_mult": 1.0,
        },
        # Electron density sources/sink (for the ne equation).
        "generic_particle_source": {
            # total particle source
            "S_tot": 0.0,
            # particle source Gaussian central location (normalized radial
            # coordinate)
            "deposition_location": 0.3,
            # particle source Gaussian width (normalized radial coordinate)
            "particle_width": 0.25,
        },
        # Ion and electron heat sources (for the temp-ion and temp-el eqs).
        "generic_ion_el_heat_source": {
            "rsource": 0.3,
            # Gaussian width in normalized radial coordinate r
            "w": 0.1,
            # total heating (including accounting for radiation) r
            "Ptot": 11.0e6,
            # electron heating fraction r
            "el_heat_fraction": 0.22,  # external power electron heating fraction (Rodriguez-Fernandez, 2020)
        },
        "fusion_heat_source": {},
        "qei_source": {
            # multiplier for ion-electron heat exchange term for sensitivity
            "Qei_mult": 1.0,
        },
    },
    "transport": {
        "transport_model": "qlknn",
        # set inner core transport coefficients (ad-hoc MHD/EM transport)
        "apply_inner_patch": True,
        "De_inner": 0.25,
        "Ve_inner": 0.0,
        "chii_inner": 1.0,
        "chie_inner": 1.0,
        "rho_inner": 0.2,  # radius below which patch transport is applied
        # set outer core transport coefficients (L-mode near edge region)
        "apply_outer_patch": True,
        "De_outer": 0.1,
        "Ve_outer": 0.0,
        "chii_outer": 2.0,
        "chie_outer": 2.0,
        "rho_outer": 0.9,  # radius above which patch transport is applied
        # allowed chi and diffusivity bounds
        "chimin": 0.05,  # minimum chi
        "chimax": 100,  # maximum chi (can be helpful for stability)
        "Demin": 0.05,  # minimum electron diffusivity
        "qlknn_params": {
            "DVeff": True,
            "include_ITG": True,  # to toggle ITG modes on or off
            "include_TEM": True,  # to toggle TEM modes on or off
            "include_ETG": True,  # to toggle ETG modes on or off
            # ensure that smag - alpha > -0.2 always, to compensate for no slab
            # modes
            "avoid_big_negative_s": True,
            # minimum |R/Lne| below which effective V is used instead of
            # effective D
            "An_min": 0.05,
            "ITG_flux_ratio_correction": 1,
        },
    },
    "stepper": {
        "stepper_type": "linear",
        "predictor_corrector": True,
        "corrector_steps": 1,
        # (deliberately) large heat conductivity for Pereverzev rule
        "chi_per": 30,
        # (deliberately) large particle diffusion for Pereverzev rule
        "d_per": 15,
        "use_pereverzev": True,
    },
    "time_step_calculator": {
        "calculator_type": "fixed",
    },
}

## Running Torax
POPSIM has a simple `run_torax` function that allows you to input a torax configuration dictionary and get out a `xr.Dataset` with the results of the simulation. Here is an example of how to run a simple simulation.

In [ ]:
import hvplot.xarray

ds = run_torax(CONFIG)

# Visualize the ne profile evolution.
ds["ne"].sel(rho_cell_norm=[0.0, 0.3, 0.5, 0.9, 1.0], method="nearest").hvplot(x="time", by="rho_cell_norm")

## Running Batches of Torax Simulations

Like most things in POPSIM, you can also use the POPSIM Torax hook to run a batch of simulations by simply providing a list of configuration dictionaries. Below, let's copy the configuration dictionary, modify the plasma current, and run a batch of simulations.

In [ ]:
from copy import deepcopy

CONFIG_NEW = deepcopy(CONFIG)
CONFIG_NEW["runtime_params"]["profile_conditions"]["Ip_tot"] = 7.0

ds = run_torax([CONFIG, CONFIG_NEW])

# Visualize the line-averaged density and plasma current evolution.
ds["ne"].integrate("rho_cell_norm").hvplot.scatter(x="time", by="simulation") + ds["Ip_profile_face"].isel(rho_face_norm=-1).hvplot.scatter(
    x="time", by="simulation"
)

## Generating Batches with Monte Carlo Sampling
We can also use the tools in `popsim.sim_utils` to define distributions on Torax config dictionaries and sample from them to generate a batch of simulations.

Let's try a "real world" task of generating a whole bunch of Torax simulations for SPARC L-mode scenarios to enable us to train a NN predictor of profile shapes.

Let's begin by loading up the baseline configuration.

In [ ]:
from popsim.interfaces.torax import get_sparc_lmode_base_config

BASE_CONFIG = get_sparc_lmode_base_config()

# Environment variables.
NSIM = 2
SAVE_SIMULATION = False
TIME_DOWNSAMPLE = 10
SEED = 0
MAX_WORKERS = 10
TIME_START = 2.0
OUTPUT_ZARR = "/tmp/torax_sims.zarr"

Now let's define some distributions to override the values in the configuration dictionary. We can then call `sample_samplers` to generate a list of configuration dictionaries to run.

**TODO: get a transport expert to improve the configuration.**

In [ ]:
import copy

import jax

from popsim.sim_utils import StaticSampler, sample_samplers

SWEEP_RANGES = {
    "Tedge": (0.9, 2.3),
    "nedge": (0.77, 4.0),
    "Ip_tot": (3.0, 8.7),
    "Zeff": (1.5, 1.51),  # Mostly just keep 1.5 for now.
    "Ptot": (0.0e6, 11e6),
    "elongation_LCFS": (1.5, 2.0),
    "Rmin": (0.4, 0.57),
    "S_tot": (0.25e21, 0.8e21),  # Particles per second. Upper bound roughly manually chosen to avoid the "kink" in the profile.
}


def make_uniform_sampler(lower, upper):
    def sampler_fn(key):
        return jax.random.uniform(key, minval=lower, maxval=upper)

    sampler = StaticSampler(sampler_fn)
    return sampler


def sample_profile_conditions(key):
    key, subkey = jax.random.split(key)
    profile_conditions = copy.deepcopy(BASE_CONFIG["runtime_params"]["profile_conditions"])

    # Using SWEEP_RANGES["Ip"] for Ip
    min_Ip, max_Ip = SWEEP_RANGES["Ip_tot"]
    profile_conditions["Ip_tot"] = jax.random.uniform(subkey, minval=min_Ip, maxval=max_Ip)

    # Using SWEEP_RANGES["Tedge"] for Tedge
    key, subkey = jax.random.split(key)
    min_Tedge, max_Tedge = SWEEP_RANGES["Tedge"]
    Tedge = jax.random.uniform(subkey, minval=min_Tedge, maxval=max_Tedge)
    profile_conditions["Tiped"] = Tedge
    profile_conditions["Teped"] = Tedge

    # Using SWEEP_RANGES["nedge"] for ne_bound_right
    key, subkey = jax.random.split(key)
    min_nedge, max_nedge = SWEEP_RANGES["nedge"]
    profile_conditions["neped"] = jax.random.uniform(subkey, minval=min_nedge, maxval=max_nedge)
    profile_conditions["ne_bound_right"] = profile_conditions["neped"]

    # Initialize the guess for the electron density profile.
    profile_conditions["ne"][0][1.0] = profile_conditions["neped"]
    profile_conditions["ne"][0][0.0] = profile_conditions["neped"] + 0.5

    return profile_conditions


# Updated configuration using SWEEP_RANGES
CONFIG_BATCH = copy.deepcopy(BASE_CONFIG)

CONFIG_BATCH["runtime_params"]["profile_conditions"] = StaticSampler(
    sampler_fn=sample_profile_conditions,
)
CONFIG_BATCH["runtime_params"]["plasma_composition"]["Zeff"] = make_uniform_sampler(*SWEEP_RANGES["Zeff"])
CONFIG_BATCH["sources"]["generic_ion_el_heat_source"]["Ptot"] = make_uniform_sampler(*SWEEP_RANGES["Ptot"])
CONFIG_BATCH["sources"]["generic_particle_source"]["S_tot"] = make_uniform_sampler(*SWEEP_RANGES["S_tot"])
CONFIG_BATCH["geometry"]["elongation_LCFS"] = make_uniform_sampler(*SWEEP_RANGES["elongation_LCFS"])
CONFIG_BATCH["geometry"]["Rmin"] = make_uniform_sampler(*SWEEP_RANGES["Rmin"])

config_cases = sample_samplers(CONFIG_BATCH, jax.random.PRNGKey(SEED), NSIM)

In [ ]:
from popsim.interfaces.torax import run_torax

run_torax(config_cases, max_workers=MAX_WORKERS, output_zarr=OUTPUT_ZARR if SAVE_SIMULATION else None)

In [ ]:
import os

import xarray as xr

from popsim import DATA_DIR

if SAVE_SIMULATION:
    ds = xr.open_zarr(OUTPUT_ZARR)
    ds = ds.sel(time=slice(TIME_START, None))
    ds = ds.isel(time=slice(None, None, TIME_DOWNSAMPLE))
    ds.to_netcdf(os.path.join(DATA_DIR, "sparc/torax_profile_predictor.nc"))